# Tarea 1

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.impute import SimpleImputer

In [2]:
datos_entrenamiento = pd.read_csv("train.csv")
datos_prueba = pd.read_csv("test.csv")

In [3]:
objetivo = datos_entrenamiento['failure']

datos_entrenamiento = datos_entrenamiento.drop(columns=['id', 'product_code', 'failure'])
ids_prueba = datos_prueba['id']
datos_prueba = datos_prueba.drop(columns=['id', 'product_code'])

In [4]:
columnas_con_nulos = [col for col in datos_entrenamiento.columns if datos_entrenamiento[col].isnull().any()]

for columna in columnas_con_nulos:
    datos_entrenamiento[f'{columna}_es_nulo'] = datos_entrenamiento[columna].isnull().astype(int)
    datos_prueba[f'{columna}_es_nulo'] = datos_prueba[columna].isnull().astype(int)

In [5]:
datos_entrenamiento = pd.get_dummies(datos_entrenamiento, drop_first=True)
datos_prueba = pd.get_dummies(datos_prueba, drop_first=True)

datos_entrenamiento, datos_prueba = datos_entrenamiento.align(datos_prueba, join='left', axis=1, fill_value=0)

In [6]:
imputador = SimpleImputer(strategy='median')

nombres_columnas = datos_entrenamiento.columns

datos_entrenamiento = pd.DataFrame(imputador.fit_transform(datos_entrenamiento), columns=nombres_columnas)
datos_prueba = pd.DataFrame(imputador.transform(datos_prueba), columns=nombres_columnas)

In [7]:
modelo_prediccion = GradientBoostingClassifier(random_state=42)
validador_cruzado = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

puntuaciones_auc = cross_val_score(
    modelo_prediccion,
    datos_entrenamiento,
    objetivo,
    cv=validador_cruzado,
    scoring='roc_auc'
)

In [8]:
print(puntuaciones_auc)
print(np.mean(puntuaciones_auc))

[0.58871195 0.58171775 0.58755838 0.57662905 0.57801306]
0.5825260374547472


### Conclusión Tarea 1: Rendimiento del Modelo de Predicción

El modelo obtiene un AUC-ROC medio aproximado de 0.58 en validación cruzada. Este rendimiento modesto es el resultado directo de un escenario de generalización estricto: el modelo se entrena con un conjunto de productos (códigos A-E) y se evalúa sobre productos completamente nuevos (códigos F-I). Al no haber solapamiento en los productos, el modelo no puede memorizar comportamientos específicos y se ve obligado a inferir basándose únicamente en los atributos y mediciones físicas, lo que representa un reto predictivo considerable.

# Tarea 2

In [9]:
datos_entrenamiento['es_prueba'] = 0
datos_prueba['es_prueba'] = 1

datos_combinados = pd.concat([datos_entrenamiento, datos_prueba], ignore_index=True)

objetivo_adversario = datos_combinados['es_prueba']
datos_adversarios = datos_combinados.drop(columns=['es_prueba'])

In [10]:
modelo_adversario = GradientBoostingClassifier(random_state=42)
validador_cruzado_adv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

puntuaciones_auc_adv = cross_val_score(
    modelo_adversario,
    datos_adversarios,
    objetivo_adversario,
    cv=validador_cruzado_adv,
    scoring='roc_auc'
)

In [11]:
print(puntuaciones_auc_adv)
print(np.mean(puntuaciones_auc_adv))

[1. 1. 1. 1. 1.]
1.0


In [12]:
modelo_adversario.fit(datos_adversarios, objetivo_adversario)

importancias = pd.DataFrame({
    'caracteristica': datos_adversarios.columns,
    'importancia': modelo_adversario.feature_importances_
}).sort_values(by='importancia', ascending=False)

print(importancias.head(10))

            caracteristica   importancia
1              attribute_2  3.948410e-01
2              attribute_3  2.582289e-01
39  attribute_1_material_8  2.278443e-01
37  attribute_0_material_7  1.107181e-01
38  attribute_1_material_6  8.367663e-03
6            measurement_3  1.277050e-13
13          measurement_10  5.004839e-14
8            measurement_5  4.926029e-14
20          measurement_17  1.004577e-14
12           measurement_9  6.445507e-15


### Conclusión Tarea 2: Diagnóstico de Deriva de Datos (Data Drift)

La validación adversarial arroja un AUC-ROC perfecto (1.0), lo que confirma la existencia de un "data drift" severo entre el conjunto de entrenamiento y el de prueba. Al analizar la importancia de las características del modelo adversario, se evidencia que este drift es de naturaleza estructural y está dominado casi en su totalidad por los atributos categóricos de los productos (`attribute_2`, `attribute_3`, etc.). Las mediciones continuas, sin embargo, mantienen distribuciones más estables. Esto explica la dificultad del modelo de la Tarea 1: se enfrenta a configuraciones de materiales que no existían en la fase de entrenamiento.

### ¿Cómo se comportaría este modelo si existe data drift? ¿Y en el caso contrario?

Para comprender el comportamiento de este modelo adversario, debemos evaluar su capacidad para distinguir la procedencia de los datos. Si existe un "data drift" o deriva de datos, el modelo detectará diferencias estadísticas y patrones divergentes entre las características de ambos conjuntos. Al poder identificar estas variaciones, logrará clasificar con éxito qué registros pertenecen al conjunto de entrenamiento y cuáles al de prueba, lo que se traducirá en un rendimiento elevado con una métrica AUC-ROC cercana a 1.0. Por el contrario, en el caso opuesto donde los conjuntos de datos son completamente homogéneos y no presentan deriva, el modelo será incapaz de encontrar una regla discriminatoria válida. Al ser estadísticamente indistinguibles, las predicciones del modelo equivaldrían a una estimación aleatoria, estancando su rendimiento en un AUC-ROC en torno a 0.5. Cabe destacar que, para garantizar la validez técnica de este análisis, se ha omitido la columna del código de producto, asegurando así que el modelo evalúe la deriva genuina presente en los atributos y mediciones de laboratorio, evitando que aprenda una separación trivial basada únicamente en la nomenclatura del producto.